In [1]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor

sys.path.append('../')
import importlib
import yaml
import torch
import os 
from pathlib import Path

from lightning_scripts.eval_jsin_transfer import SSLWordClassifier

In [2]:
## init config. Will be yaml eventually, but start as dict 
config_path = Path("model_configs/pilot_ssl_word_resnet50.yaml")
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

# Overwrite config for linear classifier 
config['num_workers'] = 4
config['hparas']['batch_size'] = 72
config['hparas']['optimizer'] = "AdamW"
config['hparas']['lr'] = 0.00001
config['data']['eval_max'] = 1



In [3]:
model_ckpt_dir = "model_checkpoints"
checkpoint_dir = Path(model_ckpt_dir) / f"{config_path.stem}/checkpoints"
ckpt_paths = sorted(checkpoint_dir.glob("*.ckpt"), key=os.path.getctime)
ckpt_path = ckpt_paths[-1] # get latest checkpoint 
print(ckpt_path)

model_checkpoints/pilot_ssl_word_resnet50/checkpoints/epoch=9-step=37500-v1.ckpt


In [4]:
module = SSLWordClassifier(config=config,
                           ckpt_path=ckpt_path,
                           layer_out='avgpool')

In [5]:
trainer = L.Trainer(devices=1)


/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [6]:
trainer.fit(module)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/loops/utilities.py:72: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name              | Type             | Params | Mode 
---------------------------------------------------------------
0 | feature_extractor | OptimizedModule  | 26.4 M | train
1 | classifier        | Linear           | 1.6 M  | train
2 | loss              | CrossEntropyLoss | 0      | train
---------------------------------------------------------------
1.6 M     Trainable params
26.4 M    Non-trainable params
28.1 M    Total params
112.269   Total estimated model params size (MB)
3         Modules in train mode
171       Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [6]:
outputs = trainer.predict(module, module.val_dataloader(), return_predictions=True)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [20]:
output_vals = torch.cat([output['accuracy'] for output in outputs])
len(output_vals)

16776

In [27]:
output_vals.mean()

tensor(0.0010)

In [31]:

output_vals.std(unbiased=True) / (output_vals.size(0) ** 0.5)

tensor(0.0002)

In [43]:
import pickle
with open('eval_jsin_results/ssl_barlow_word_resnet50_hparam_set_0_linear_eval_jsin.pkl', 'rb') as handle:

    results = pickle.load(handle)

In [44]:
results

{'mean_acc': tensor(0.0018),
 'std_acc': tensor(0.0423),
 'sem_acc': tensor(0.0003)}